In [ ]:
!pip install ultralytics lap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 8.4 MB/s eta 0:00:00


In [ ]:
"""
Assignment 9: Apply object detection from CCTV Footage.

Parts:
  A. YOLOv8 object detection on every frame of a video (people, cars, bikes ...)
  B. Object tracking (persistent IDs) + unique object counting
  C. Region-of-interest (restricted zone) alert + virtual line crossing counter
  D. Annotated output video + CSV detection log + summary chart
  E. (Fallback) Motion detection with OpenCV background subtraction - no model needed

Install:
    pip install ultralytics opencv-python numpy pandas matplotlib

Usage:
    python assignment_09_cctv_object_detection.py --source cctv.mp4
    python assignment_09_cctv_object_detection.py --source 0            # webcam / live camera
    python assignment_09_cctv_object_detection.py --source rtsp://user:pass@ip:554/stream
    python assignment_09_cctv_object_detection.py --source cctv.mp4 --mode motion

Note: the YOLO weights file (yolov8n.pt) is downloaded automatically on first run.
Press 'q' in the video window to stop.
"""

import argparse
import csv
import os
import subprocess
import sys
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------
# COCO class ids we care about for CCTV: person, bicycle, car, motorcycle, bus, truck
TARGET_CLASSES = [0, 1, 2, 3, 5, 7]
CONF_THRESHOLD = 0.4

# Restricted zone polygon (x, y) in pixels - edit to match your camera view
ZONE_POLYGON = np.array([[100, 200], [500, 200], [500, 450], [100, 450]], np.int32)

# Virtual counting line: (x1, y1) -> (x2, y2). Objects crossing it are counted.
COUNT_LINE = ((50, 350), (900, 350))


# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------
def point_in_zone(point, polygon):
    """True if (x, y) lies inside the polygon."""
    return cv2.pointPolygonTest(polygon, (float(point[0]), float(point[1])), False) >= 0


def side_of_line(point, line):
    """Sign tells which side of the line the point is on."""
    (x1, y1), (x2, y2) = line
    return np.sign((x2 - x1) * (point[1] - y1) - (y2 - y1) * (point[0] - x1))


def open_video(source):
    src = int(source) if str(source).isdigit() else source
    cap = cv2.VideoCapture(src)
    if not cap.isOpened():
        raise IOError(f"Cannot open video source: {source}")
    return cap


def make_writer(cap, path):
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    return cv2.VideoWriter(path, fourcc, fps, (w, h)), fps


# ==================================================================
# PARTS A-D: YOLO detection + tracking + zone alert + line counting
# ==================================================================
def run_yolo(source, output="cctv_output.mp4", log_csv="detections_log.csv",
             show=True, weights="yolov8n.pt"):
    from ultralytics import YOLO

    model = YOLO(weights)
    names = model.names
    cap = open_video(source)
    writer, fps = make_writer(cap, output)

    seen_ids = defaultdict(set)         # class name -> set of unique track ids
    last_side = {}                      # track id -> side of the counting line
    crossed = set()                     # ids that already crossed the line
    line_count = 0
    alert_ids = set()                   # ids that entered the restricted zone
    frame_counts = []                   # per-frame object counts for the chart
    log_rows = []

    frame_no = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_no += 1

        # persist=True keeps the same ID for an object across frames
        result = model.track(frame, persist=True, classes=TARGET_CLASSES,
                             conf=CONF_THRESHOLD, verbose=False)[0]

        counts_this_frame = defaultdict(int)

        # Draw zone and counting line
        cv2.polylines(frame, [ZONE_POLYGON], True, (0, 255, 255), 2)
        cv2.line(frame, COUNT_LINE[0], COUNT_LINE[1], (255, 0, 0), 2)

        if result.boxes is not None and len(result.boxes) > 0:
            boxes = result.boxes
            xyxy = boxes.xyxy.cpu().numpy().astype(int)
            cls_ids = boxes.cls.cpu().numpy().astype(int)
            confs = boxes.conf.cpu().numpy()
            ids = (boxes.id.cpu().numpy().astype(int)
                   if boxes.id is not None else [None] * len(xyxy))

            for (x1, y1, x2, y2), c, conf, tid in zip(xyxy, cls_ids, confs, ids):
                label = names[c]
                counts_this_frame[label] += 1
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2     # centre of the box
                colour = (0, 255, 0)

                if tid is not None:
                    seen_ids[label].add(int(tid))

                    # --- restricted zone alert (people only) ---
                    if label == "person" and point_in_zone((cx, y2), ZONE_POLYGON):
                        colour = (0, 0, 255)
                        alert_ids.add(int(tid))
                        cv2.putText(frame, "ALERT: restricted zone", (x1, y1 - 25),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

                    # --- line crossing counter ---
                    side = side_of_line((cx, cy), COUNT_LINE)
                    if tid in last_side and side != last_side[tid] and tid not in crossed:
                        crossed.add(tid)
                        line_count += 1
                    last_side[tid] = side

                cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
                tag = f"{label} {conf:.2f}" + (f" #{tid}" if tid is not None else "")
                cv2.putText(frame, tag, (x1, y1 - 7), cv2.FONT_HERSHEY_SIMPLEX,
                            0.55, colour, 2)
                cv2.circle(frame, (cx, cy), 3, colour, -1)

                log_rows.append([frame_no, round(frame_no / fps, 2), label,
                                 None if tid is None else int(tid),
                                 round(float(conf), 3), x1, y1, x2, y2])

        # On-screen dashboard
        y_text = 25
        for label, n in counts_this_frame.items():
            cv2.putText(frame, f"{label}: {n}", (10, y_text),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            y_text += 25
        cv2.putText(frame, f"Line crossings: {line_count}", (10, y_text),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 200, 0), 2)

        frame_counts.append({"frame": frame_no, **counts_this_frame})
        writer.write(frame)

        if show:
            cv2.imshow("CCTV Object Detection (press q to quit)", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    cap.release()
    writer.release()
    cv2.destroyAllWindows()

    # ---------------- Save log and print summary ----------------
    with open(log_csv, "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["frame", "time_sec", "class", "track_id",
                     "confidence", "x1", "y1", "x2", "y2"])
        wr.writerows(log_rows)

    print("\n" + "=" * 55)
    print("DETECTION SUMMARY")
    print("=" * 55)
    print(f"Frames processed          : {frame_no}")
    for label, ids in seen_ids.items():
        print(f"Unique {label:<12} tracked : {len(ids)}")
    print(f"Objects crossing the line : {line_count}")
    print(f"People entering the zone  : {len(alert_ids)}")
    print(f"Annotated video saved to  : {os.path.abspath(output)}")
    print(f"Detection log saved to    : {os.path.abspath(log_csv)}")

    # ---------------- Chart: objects per frame ----------------
    df = pd.DataFrame(frame_counts).fillna(0)
    if not df.empty:
        df.set_index("frame").plot(figsize=(10, 4.5))
        plt.title("Number of Detected Objects per Frame")
        plt.xlabel("Frame")
        plt.ylabel("Count")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig("objects_per_frame.png", dpi=120)
        plt.show()


# ==================================================================
# PART E: Fallback - motion detection (no deep learning model needed)
# ==================================================================
def run_motion_detection(source, output="cctv_motion_output.mp4", show=True,
                         min_area=800):
    cap = open_video(source)
    writer, _ = make_writer(cap, output)
    subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50,
                                                    detectShadows=True)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    frame_no, motion_frames = 0, 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_no += 1

        mask = subtractor.apply(frame)
        _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)   # drop shadows
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.dilate(mask, kernel, iterations=2)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        moving = 0
        for cnt in contours:
            if cv2.contourArea(cnt) < min_area:
                continue
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            moving += 1

        if moving:
            motion_frames += 1
            cv2.putText(frame, f"MOTION DETECTED ({moving})", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

        writer.write(frame)
        if show:
            cv2.imshow("Motion Detection (press q to quit)", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    cap.release()
    writer.release()
    cv2.destroyAllWindows()
    print(f"\nFrames processed: {frame_no} | frames with motion: {motion_frames}")
    print("Saved:", os.path.abspath(output))


# ------------------------------------------------------------------
# Main
# ------------------------------------------------------------------
def in_notebook():
    """True when running inside Colab / Jupyter."""
    return "google.colab" in sys.modules or "ipykernel" in sys.modules


def play_in_notebook(video_path):
    """Convert to browser-friendly H.264 and play inside the notebook."""
    from IPython.display import HTML, display
    from base64 import b64encode

    fixed = video_path.replace(".mp4", "_h264.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", video_path,
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", fixed], check=True)
    data = b64encode(open(fixed, "rb").read()).decode()
    display(HTML(f'<video width="720" controls>'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Object detection on CCTV footage")
    parser.add_argument("--source", default="cctv.mp4",
                        help="video file path, camera index (0) or RTSP url")
    parser.add_argument("--mode", choices=["yolo", "motion"], default="yolo")
    parser.add_argument("--weights", default="yolov8n.pt",
                        help="YOLO weights (yolov8n.pt = fastest, yolov8s/m = more accurate)")
    parser.add_argument("--no-show", action="store_true",
                        help="do not open a preview window (useful on servers / Colab)")
    # parse_known_args ignores extra arguments such as Colab's "-f kernel.json"
    args, _ = parser.parse_known_args()

    # cv2.imshow does not work in Colab/Jupyter, so turn the preview off there
    show = not args.no_show and not in_notebook()

    if args.mode == "yolo":
        run_yolo(args.source, show=show, weights=args.weights)
        out_file = "cctv_output.mp4"
    else:
        run_motion_detection(args.source, show=show)
        out_file = "cctv_motion_output.mp4"

    if in_notebook():
        play_in_notebook(out_file)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


OSError: Cannot open video source: cctv.mp4